# 章节实践解答（05.07）

本解答对应章节 05.07_chapter_test.ipynb 中的综合编程实践题。

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer


def build_lora_model(model):
    lora_config = LoraConfig(
        r=8, lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM")
    return get_peft_model(model, lora_config)


def train(model, tokenizer, train_data, eval_data):
    args = SFTConfig(
        output_dir="./sft_output",
        max_steps=500,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        bf16=True,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        save_strategy="steps",
        save_steps=50,
    )
    trainer = SFTTrainer(
        model=model, args=args,
        train_dataset=train_data, eval_dataset=eval_data)
    trainer.train()

In [ ]:
def chat_infer(query, tokenizer, model, max_new_tokens=256):
    prompt = f"<|user|>\n{query}\n<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.7, top_p=0.9)
    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return answer.split("<|end|>")[0].strip()


# 评估维度：专业度 / 共情度 / 建议性 / 安全性

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px;">点击：查看/折叠代码说明</summary>
  <div style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li>LoRA 只训练 q_proj/v_proj 上的低秩矩阵，可训练参数约 393 万，占总参数 0.057%。</li>
      <li>训练时开启 bf16 与 gradient accumulation，单卡 NPU 即可完成 7B 模型微调。</li>
    </ul>
  </div>
</details>